# Использование моделей с Hugging Face

Обучать собственные модели машинного обучения - это круто. Но всегда есть предел вычислительных мощностей и объемов данных. И тогда на помощь приходит сообщество датасатанистов!

Разработчики крупных компаний совместно с энтузиастами постоянно совершенствуют классические архитектуры модели, обучая их для решения различных задач. Так повелось, что домом для сообщества датасатанистов стал ресурс [HuggingFace](https://huggingface.co/), поддерживаемый одноименной компанией.

Сегодня мы рассмотрим, как можно использовать этот ресурс для решения своих задач на примере задачи классификации предложений.

## Библиотека Transformers и пайплайны

Для использования моделей с HuggingFace используется библиотека [transformers](https://pypi.org/project/transformers/). По умолчанию в Colab эта библиотека не установлена (а зря).

In [1]:
! pip install -U accelerate -q
! pip install -U transformers -q

In [2]:
import transformers
transformers.__version__

'4.34.1'

Самый простой и распространенный вариант использования моделей - инференс с применением [пайплайнов](https://huggingface.co/docs/transformers/main_classes/pipelines) для различных задач.

In [3]:
from transformers import pipeline

In [4]:
classifier = pipeline('text-classification', model='tinkoff-ai/response-toxicity-classifier-base')

In [5]:
pred = classifier('ты дурачек')

In [6]:
pred

[{'label': 'toxic', 'score': 0.9836204051971436}]

В таком варианте мы просто и быстро задействовали готовую модель, ничего не настраивая. При этом пайплайны можно настраивать. Рассмотрим это на примере генерации текста (реализация GPT 3 от Сбера).

In [7]:
generator = pipeline('text-generation', model='ai-forever/rugpt3small_based_on_gpt2')

In [8]:
res = generator('Сегодня утром я пришел на пару к Гуненкову, а затем ', max_length=32)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [9]:
res

[{'generated_text': 'Сегодня утром я пришел на пару к Гуненкову, а затем  в  магазин, где  купил  себе  новый  костюм  и  галстук.'}]

Например, я могу задать максимальное количество токенов (считая промпт).

In [10]:
res = generator('Что такое кошка?', max_length=128)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [11]:
res

[{'generated_text': 'Что такое кошка?\nКошка - это кошка, которая живет в доме.\n\nКакие есть способы, чтобы не было проблем с кожей?\nПопробуйте крем для лица.\n\nКакие есть способы, чтобы не было проблем с кожей?\nПопробуйте крем для лица.\n\nКакие есть способы, чтобы не было проблем с кожей?\nПопробуйте крем для лица.\n\nКакие есть способы, чтобы не было проблем с кожей?\nПопробуйте крем для лица.\n\nКакие есть способы, чтобы не было проблем с кожей?\nПопробуйте крем для лица.\n\n'}]

Это модель начального уровня, она пытается, но справляется не очень.

## Модели и токенайзеры

Работа с пайплайнами закрывает 99% кейсов инференса готовых моделей, однако нам следует отдельно рассмотреть две основные сущности библиотеки transformers - модели и токенайзеры. Рассматривать их будем на модели для классификации текстов.

In [12]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

Каждая модель, которая есть в Hugging Face состоит из двух частей - собственно сама модель и токенайзер.

In [13]:
model = AutoModelForSequenceClassification.from_pretrained('tinkoff-ai/response-toxicity-classifier-base')
tokenizer = AutoTokenizer.from_pretrained('tinkoff-ai/response-toxicity-classifier-base')

In [14]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(100793, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [15]:
tokenizer

BertTokenizerFast(name_or_path='tinkoff-ai/response-toxicity-classifier-base', vocab_size=100792, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]', 'additional_special_tokens': ['[RESPONSE_TOKEN]']}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100792: AddedToken("[

Посмотрим сначала на токенайзер.

In [16]:
t = tokenizer.tokenize('Меня зовут Миша')
t

['Меня', 'зовут', 'Миша']

In [17]:
i = tokenizer.convert_tokens_to_ids(t)
i

[4211, 7881, 28827]

In [18]:
d = tokenizer.decode(i)
d

'Меня зовут Миша'

Мы можем комбинировать библиотеки transformers и PyTorch (можно использовать и TensorFlow).

In [19]:
texts = ['Я тебя обожаю', 'Я тебя ненавижу']
tokenized = tokenizer(texts, padding=True, return_tensors='pt')
tokenized

{'input_ids': tensor([[  101,   478,  1425, 28441,   102],
        [  101,   478,  1425, 18480,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1]])}

In [20]:
import torch

In [21]:
with torch.no_grad():
  pred = model(**tokenized)
  print(pred)

SequenceClassifierOutput(loss=None, logits=tensor([[ 5.7812, -1.7483, -2.5264, -0.4957],
        [-0.1533, -3.3699, -0.9324,  3.8884]]), hidden_states=None, attentions=None)


В этом примере мы отключили AUTOGRAD в PyTorch, что позволяет увеличить скорость. На выходе модели мы получаем логиты, далее можем применить softmax и т. д.

Таким образом, модели с HuggingFace могут быть использованы для получения эмбеддингов, которые мы далее будем использовать в своих моделях PyTorch.

Для того, чтобы каждый раз не скачивать модели с HuggingFace мы можем сохранить их на диск в папку (отдельно токенайзер и саму модель).

In [22]:
tokenizer.save_pretrained('mymodel')
model.save_pretrained('mymodel')

In [23]:
model2 = AutoModelForSequenceClassification.from_pretrained('mymodel')
tokenizer2 = AutoTokenizer.from_pretrained('mymodel')

In [24]:
with torch.no_grad():
  pred = model2(**tokenized)
  print(pred)

SequenceClassifierOutput(loss=None, logits=tensor([[ 5.7812, -1.7483, -2.5264, -0.4957],
        [-0.1533, -3.3699, -0.9324,  3.8884]]), hidden_states=None, attentions=None)


## Fine-tuning

Библиотека transformers предоставляет нам удобные возможности для проведения дообучения моделей. При этом для выполнения fine-tuning нам необходима сама модель и токенайзер.

Следует помнить, что мы можем взять токенайзер с любой из предварительно обученных моделей. А можем и [создать свой собственный](https://huggingface.co/learn/nlp-course/chapter6/8?fw=pt), с помощью библиотеки tokenizers (она является зависимостью для transformers и устанавливается вместе с ней).

Загрузим [данные](https://drive.google.com/file/d/1gvHtpqh0eswHrmqhTelKWyN2AfWeUJ5O/view?usp=sharing).

In [25]:
import pandas as pd

In [26]:
df = pd.read_csv('AirlineTweets.csv')

In [27]:
df

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14635,569587686496825344,positive,0.3487,NaN,0.0000,American,NaN,KristenReenders,NaN,0,@AmericanAir thank you we got on a different f...,NaN,2015-02-22 12:01:01 -0800,NaN,NaN
14636,569587371693355008,negative,1.0000,Customer Service Issue,1.0000,American,NaN,itsropes,NaN,0,@AmericanAir leaving over 20 minutes Late Flig...,NaN,2015-02-22 11:59:46 -0800,Texas,NaN
14637,569587242672398336,neutral,1.0000,NaN,NaN,American,NaN,sanyabun,NaN,0,@AmericanAir Please bring American Airlines to...,NaN,2015-02-22 11:59:15 -0800,"Nigeria,lagos",NaN
14638,569587188687634433,negative,1.0000,Customer Service Issue,0.6659,American,NaN,SraJackson,NaN,0,"@AmericanAir you have my money, you change my ...",NaN,2015-02-22 11:59:02 -0800,New Jersey,Eastern Time (US & Canada)


In [28]:
df = df[['airline_sentiment', 'text']]
df

,airline_sentiment,text
0,neutral,@VirginAmerica What @dhepburn said.
1,positive,@VirginAmerica plus you've added commercials t...
2,neutral,@VirginAmerica I didn't today... Must mean I n...
3,negative,@VirginAmerica it's really aggressive to blast...
4,negative,@VirginAmerica and it's a really big bad thing...
...,...,...
14635,positive,@AmericanAir thank you we got on a different f...
14636,negative,@AmericanAir leaving over 20 minutes Late Flig...
14637,neutral,@AmericanAir Please bring American Airlines to...
14638,negative,"@AmericanAir you have my money, you change my ..."


In [29]:
import numpy as np

In [30]:
np.unique(df['airline_sentiment'])

array(['negative', 'neutral', 'positive'], dtype=object)

Будем решать задачу классификации с тремя классами.

In [31]:
df['target'] = df['airline_sentiment'].map({ 'positive': 1, 'negative': 0, 'neutral': 2})
df

<ipython-input-31-0e8d8a250b52>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['target'] = df['airline_sentiment'].map({ 'positive': 1, 'negative': 0, 'neutral': 2})


,airline_sentiment,text,target
0,neutral,@VirginAmerica What @dhepburn said.,2
1,positive,@VirginAmerica plus you've added commercials t...,1
2,neutral,@VirginAmerica I didn't today... Must mean I n...,2
3,negative,@VirginAmerica it's really aggressive to blast...,0
4,negative,@VirginAmerica and it's a really big bad thing...,0
...,...,...,...
14635,positive,@AmericanAir thank you we got on a different f...,1
14636,negative,@AmericanAir leaving over 20 minutes Late Flig...,0
14637,neutral,@AmericanAir Please bring American Airlines to...,2
14638,negative,"@AmericanAir you have my money, you change my ...",0


Очень важно - нам необходимо переименовать колонку с целевым признаком в label!

In [32]:
df = df[['text', 'target']]
df.columns = ['sentence', 'label']
df.to_csv('data.csv', index=False)

Будем использовать библиотеку datasets (думаю, вы уже поняли, что ребята из HuggingFace не парятся по части именования своих библиотек).

In [33]:
! pip install datasets -q

In [34]:
from datasets import load_dataset

In [35]:
raw_dataset = load_dataset('csv', data_files = 'data.csv')

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [36]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 14640
    })
})

У местных датасетов есть свой вариант train_test_split!

In [37]:
split = raw_dataset['train'].train_test_split(test_size=0.2, seed=42)

In [38]:
split

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 11712
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 2928
    })
})

Загрузим токенайзер от первой попавшейся модели для английского языка (причем в данный момент не важно, от какой именно).

In [39]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

In [40]:
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert-base-uncased-finetuned-sst-2-english', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [41]:
tokenizer('hello')

{'input_ids': [101, 7592, 102], 'attention_mask': [1, 1, 1]}

In [42]:
def tokenize(batch):
  return tokenizer(batch['sentence'], padding = True)

In [43]:
tokenized_dataset = split.map(tokenize)

Map:   0%|          | 0/11712 [00:00<?, ? examples/s]

Map:   0%|          | 0/2928 [00:00<?, ? examples/s]

In [44]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 11712
    })
    test: Dataset({
        features: ['sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 2928
    })
})

In [45]:
from transformers import Trainer, TrainingArguments

Указываем, сколько нам нужно классов (вместо 2 будет 3).

In [68]:
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english', ignore_mismatched_sizes=True, num_labels=3)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased-finetuned-sst-2-english and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [69]:
training_args = TrainingArguments(output_dir='training_dir',
                                  evaluation_strategy='epoch',
                                  save_strategy='epoch',
                                  num_train_epochs=3,
                                  per_device_train_batch_size=16,
                                  per_device_eval_batch_size=64,
                                  )

- output_dir - папка, куда будут сохраняться результаты обучения,
- save_strategy - через какой период будет сохраняться модель
- evaluation_strategy - через какой период будет производиться оценка качества
- num_train_epochs - сколько эпох будем дообучать
- per_device_train_batch_size - batch_size при обучении
- per_device_eval_batch_size - batch_size при тестировании

Более подробно ознакомиться с Trainer и TrainerArguments можно [здесь](https://huggingface.co/docs/transformers/main_classes/trainer).

In [48]:
from sklearn.metrics import f1_score, confusion_matrix

In [67]:
def compute_metrics(logits_and_labels):
  logits, labels = logits_and_labels
  predictions = np.argmax(logits, axis=-1) # -1 - это последняя размерность
  f1 = f1_score(labels, predictions, average = 'macro')
  return {'f1_score': f1}

Инициализируем Trainer.

In [70]:
trainer = Trainer(model,
                  training_args,
                  train_dataset = tokenized_dataset['train'],
                  eval_dataset = tokenized_dataset['test'],
                  tokenizer=tokenizer,
                  compute_metrics=compute_metrics)

Начинаем обучение. Предупреждение: обучать трансформеры без GPU - очень плохая затея! CUDA ускоряет обучение в 50 раз (в моем случае).

Если в окружении используется CUDA, то при вызове метода train() будет использоваться GPU.

In [71]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Score
1,0.529400,0.423711,0.776734
2,0.365900,0.452166,0.797383
3,0.162600,0.581531,0.795621


TrainOutput(global_step=2196, training_loss=0.31562944752705335, metrics={'train_runtime': 225.1059, 'train_samples_per_second': 156.087, 'train_steps_per_second': 9.755, 'total_flos': 409786926367680.0, 'train_loss': 0.31562944752705335, 'epoch': 3.0})

Спросите почему такое качество классификации? Потому что в исходном датасете присутствует дисбаланс классов. Даже трансформеры здесь бессильны, увы. Поэтому не забываем про балансировку.

In [72]:
p = pipeline('text-classification', model = 'training_dir/checkpoint-2196', device=0) # используем GPU

In [73]:
p('normal'), p('super'), p('bad')

([{'label': 'LABEL_2', 'score': 0.6665729880332947}],
 [{'label': 'LABEL_1', 'score': 0.7902649641036987}],
 [{'label': 'LABEL_0', 'score': 0.9975086450576782}])

In [74]:
split['test']['sentence']

['@JetBlue thank you!',
 '@united thanks',
 "@SouthwestAir Thank you. The email about RR pts &amp; how to gain more wasn't clear so all of mine r gone. Anything u can do is appreciated.",
 "@AmericanAir Yes, thanks I found those, didn't see the gray tab at first :)",
 "@SouthwestAir my bday was yesterday and my girl gave birth to my first born and I couldn't get a courtesy flight change from Lynn... THANKS!",
 '@united spends 20 minutes beating up a lady about her bag fees.  Random stranger comes up and pays the fee.  #Randomactsofcorporategreed',
 '@VirginAmerica @TTINAC11 I DM you',
 '@united 4 people MCI to DEL. Preferably with Jet Airways EWR-BRU-DEL segment. Anytime in December',
 '@SouthwestAir Deborah helped me💁',
 '@JetBlue Got flight reschedule to flight form PIT to FLL and our flight had mechanical problems so we were delayed an hour Missed my connect',
 "@united A change fee of $200. Same exact flight from ATL-HOU is $165. Why a charge of $574? The math doesn't add up.",
 '@

In [75]:
predictions = p(split['test']['sentence'])
predictions

[{'label': 'LABEL_1', 'score': 0.9871034026145935},
 {'label': 'LABEL_1', 'score': 0.8989311456680298},
 {'label': 'LABEL_0', 'score': 0.9424934983253479},
 {'label': 'LABEL_1', 'score': 0.9286091923713684},
 {'label': 'LABEL_2', 'score': 0.8175082802772522},
 {'label': 'LABEL_0', 'score': 0.998060405254364},
 {'label': 'LABEL_2', 'score': 0.9854869842529297},
 {'label': 'LABEL_2', 'score': 0.9772600531578064},
 {'label': 'LABEL_1', 'score': 0.9919752478599548},
 {'label': 'LABEL_0', 'score': 0.9983717799186707},
 {'label': 'LABEL_0', 'score': 0.995211660861969},
 {'label': 'LABEL_0', 'score': 0.9956552982330322},
 {'label': 'LABEL_1', 'score': 0.9916003942489624},
 {'label': 'LABEL_0', 'score': 0.9984511137008667},
 {'label': 'LABEL_2', 'score': 0.9895607233047485},
 {'label': 'LABEL_1', 'score': 0.937038004398346},
 {'label': 'LABEL_0', 'score': 0.9878172278404236},
 {'label': 'LABEL_0', 'score': 0.9983466863632202},
 {'label': 'LABEL_0', 'score': 0.6409280896186829},
 {'label': 'LAB

In [76]:
def get_label(d):
  return int(d['label'].split('_')[1])
predictions = [get_label(d) for d in predictions]

In [77]:
predictions[0]

1

In [78]:
f1_score(split['test']['label'], predictions, average = 'macro')

0.7956213880718175

Мы с вами рассмотрели дообучение модели для задачи классификации. Рассмотреть весь функционал библиотеки transformers и ее зависимостей (таких как datasets, tokenizers и др.) невозможно в рамках курса, да и не нужно.

Если вам требуется выполнить дообучение моделей - вы без проблем основываясь на данном примере и информации из интернета сможете его произвести.

А так чаще всего вы будете пользоваться предварительно обученными моделями с HuggingFace - их настолько много, что практически любую прикладную задачу можно решить.